# MR-LSTM / MR-GRU — Tuning Run (v2)

**Goal: better metrics.** Version 1 established *that* the recurrent encoder wins as context
grows. It did not tune anything — `hidden_units=8`, per-column scaling and a flat learning rate
were all inherited from MRINN and never questioned. This notebook tests them.

### What v1 left on the table

| lever | v1 setting | why it is suspect |
|---|---|---|
| **hidden width** | `H=8`, inherited | MR-GRU can reach `H=11` (7,921 params) and stay under the 8,900 MLP baseline — **+72% capacity for free** |
| **feature scaling** | per column | at `T=32` one signal gets 32 different medians and IQRs, warping the time axis a recurrent encoder reads |
| **learning rate** | flat 1e-3, 50 epochs | no schedule, and the best epoch was never logged, so we do not know if 50 is enough |
| **depth** | 1 layer | 2 layers at `H=6` costs ~7,000 params — affordable, never tried |
| **statistical power** | 1 seed | seed spread is ±0.11–0.26 AQL, so v1 could not *detect* an improvement smaller than its headline gap |

### Two design decisions

**Tuning happens at `T=32`, not `T=64`.** The two are statistically indistinguishable
(0.015 AQL apart for MR-GRU, against ±0.11–0.26 seed noise) but `T=32` costs **54%** as much.
That roughly doubles how many configurations fit in the budget. The winning configuration is
then confirmed at `T=64`.

**Every comparison is run over multiple seeds.** A single-seed result cannot distinguish a real
gain from initialisation luck at this margin, so the queue always pairs a change with its own
baseline across the same seeds.

### Budget

Priority-ordered queue, ~7.5 h, **resumable**. Each run is written to disk the moment it
finishes and the queue skips anything already recorded. The blocks are ordered by expected
value, so a session that dies early still answers the most important questions.

> **Kaggle, CPU session, *Save & Run All*.** 7.5 h of work inside the 12 h limit.
> Do not use a GPU: these models are far too small to benefit, and the CPU session's
> 30 GB RAM is worth more.

---
> Base model: Yu et al., *A Market-Rule-Informed Neural Network for Efficient Imbalance
> Electricity Price Forecasting*, Adv. Eng. Informatics 76 (2026) 105083 —
> [github.com/runyao-yu/MRINN](https://github.com/runyao-yu/MRINN)

## 1. Environment, paths and dependencies

Nothing to install. Detects Kaggle / Colab / local and searches recursively for the dataset.

In [ ]:
import os, sys, gc, json, time, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")

ON_KAGGLE = Path("/kaggle/input").exists()
ON_COLAB  = "google.colab" in sys.modules
ENV = "Kaggle" if ON_KAGGLE else ("Colab" if ON_COLAB else "Local")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from keras import ops as K
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(a, b): return float(np.sqrt(mean_squared_error(a, b)))

if ON_KAGGLE:  OUT = Path("/kaggle/working")
elif ON_COLAB: OUT = Path("/content/drive/MyDrive/mrlstm_v2") if Path("/content/drive").exists() else Path("/content/mrlstm_v2")
else:          OUT = Path("./tuning_results")
OUT.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUT / "tuning_results.csv"
PRED_NPZ    = OUT / "tuning_predictions.npz"
CKPT_DIR    = OUT / "ckpt"; CKPT_DIR.mkdir(exist_ok=True)

DATA_PATH_OVERRIDE = ""

def _find_dataset():
    if DATA_PATH_OVERRIDE and Path(DATA_PATH_OVERRIDE).exists():
        return Path(DATA_PATH_OVERRIDE)
    for p in (Path("MRINN/Data/imbalance_data.csv"),
              Path("external/MRINN/Data/imbalance_data.csv"),
              Path("../mrinn/MRINN/Data/imbalance_data.csv"),
              Path("imbalance_data.csv")):
        if p.exists():
            return p
    roots = [Path("/kaggle/input"), Path("/content"), Path("./data")]
    csvs = [c for r in roots if r.exists() for c in r.rglob("*.csv")]
    named = [c for c in csvs if "imbalance" in c.name.lower()]
    if named: return named[0]
    if len(csvs) == 1: return csvs[0]
    return None

DATA_PATH = _find_dataset()
if DATA_PATH is None and not ON_KAGGLE:
    subprocess.run(["git","clone","--depth","1","https://github.com/runyao-yu/MRINN"], check=False)
    DATA_PATH = _find_dataset()
if DATA_PATH is None:
    ki = Path("/kaggle/input")
    listing = ("\n".join(f"    {p}" for p in sorted(ki.rglob("*"))[:40])
               if ki.exists() and any(ki.iterdir()) else "    (empty - no dataset attached)")
    raise FileNotFoundError(
        f"imbalance_data.csv not found.\n\nContents of /kaggle/input:\n{listing}\n\n"
        "Attach your dataset via Input -> '+ Add Input', or set DATA_PATH_OVERRIDE.")

_kv = tuple(int(x) for x in keras.__version__.split(".")[:2])
assert _kv >= (3, 0), f"needs Keras 3 (TF >= 2.16); this image has Keras {keras.__version__}"

print(f"environment : {ENV}")
print(f"tensorflow  : {tf.__version__}   keras {keras.__version__}")
print(f"data        : {DATA_PATH}  ({DATA_PATH.stat().st_size/1e6:.1f} MB)")
print(f"results     : {OUT}")

## 2. Reproducibility

Seeds are now an experimental variable, not a fixed constant — every configuration is run
across several so that a change can be distinguished from initialisation luck.

In [ ]:
import random

def set_random_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

set_random_seed(42)
print("seeding utility ready")

## 3. Data pipeline

Ported unchanged from MRINN's `library_imbalance/data.py`.

Splits are **chronological, never shuffled** — adjacent 15-minute intervals are near
duplicates, so random splitting would leak badly. Lag columns are built **within each
split separately**, so a test row's history never reaches back into validation data, and
the scalers are fit on **train only**.

> **Both scalers are available here, and the choice is now an experimental variable.**
> `scale_data` fits a RobustScaler per *column*, so at `T=32` one signal receives 32
> different medians and IQRs. `scale_data_shared_lags` fits one per *signal*. Version 1
> held this fixed at per-column to keep the encoder ablation clean; that ablation is
> finished, so v2 measures the difference directly.

In [ ]:
FEATS_PRICES = ["P_aFRR_pos", "P_mFRR_pos", "P_aFRR_neg", "P_mFRR_neg",
                "P_aFRR_pos_MOL", "P_aFRR_neg_MOL",
                "P_ID15_nemo", "P_ID60_nemo", "P_DA_nemo"]
FEATS_CAPACITIES = ["L_ID15", "L_ID60", "L_DA"]
FEATS_VOLUME = ["system_imbalance", "E_aFRR_pos", "E_mFRR_pos", "E_aFRR_neg", "E_mFRR_neg"]
FEATS = FEATS_PRICES + FEATS_CAPACITIES + FEATS_VOLUME
LABEL = ["imbalance_price"]

TRAIN_RANGE = ("2022-01-01 00:00:00+00:00", "2025-05-01 00:00:00+00:00")
VAL_RANGE   = ("2025-05-01 00:00:00+00:00", "2025-09-01 00:00:00+00:00")
TEST_RANGE  = ("2025-09-01 00:00:00+00:00", "2026-01-01 00:00:00+00:00")

TIME_COL = "Time [UTC] start"


def load_data(path):
    df = pd.read_csv(path, parse_dates=[TIME_COL, "Time [UTC] end"])
    df.rename(columns={"P_VoAA_pos": "P_aFRR_pos_MOL",
                       "P_VoAA_neg": "P_aFRR_neg_MOL"}, inplace=True)
    df["L_DA"] = 0          # pinned to zero upstream; its encoder gets pruned
    return df


def split_data(df, train_range, val_range, test_range):
    def m(r):
        s, e = pd.to_datetime(r[0]), pd.to_datetime(r[1])
        return (df[TIME_COL] >= s) & (df[TIME_COL] < e)      # half-open
    return df.loc[m(train_range)].copy(), df.loc[m(val_range)].copy(), df.loc[m(test_range)].copy()


def make_shifts(df, cols, lags):
    # build every lag column at once: inserting one at a time fragments the frame,
    # which is painful at T=96 (17 x 96 = 1,632 columns)
    new = {f"{c}_lag{L}": df[c].shift(L) for c in cols if c in df.columns for L in lags}
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)


def shift_data(df_train, df_val, df_test, target_col, feature_cols, lags):
    lag_cols = [f"{c}_lag{L}" for c in feature_cols for L in lags]
    needed = lag_cols + [target_col, TIME_COL]
    out = []
    for d in (df_train, df_val, df_test):
        s = make_shifts(d, feature_cols, lags)
        s = s.loc[:, ~s.columns.duplicated(keep="first")]
        out.append(s[needed].dropna().copy())
    return out[0], out[1], out[2], lag_cols


def scale_data(tr, va, te, feature_names, target_col):
    Xtr, Xva, Xte = (f[feature_names].to_numpy(float) for f in (tr, va, te))
    ytr, yva, yte = (f[target_col].to_numpy(float).reshape(-1, 1) for f in (tr, va, te))

    x_scaler = RobustScaler().fit(Xtr)          # fit on TRAIN only
    y_scaler = RobustScaler().fit(ytr)

    mk = lambda a: pd.DataFrame(x_scaler.transform(a), columns=feature_names)
    mky = lambda a: pd.DataFrame(y_scaler.transform(a).ravel(), columns=target_col)
    return mk(Xtr), mk(Xva), mk(Xte), mky(ytr), mky(yva), mky(yte), y_scaler


def scale_data_shared_lags(tr, va, te, feature_names, target_col):
    # One scaler per SIGNAL instead of per COLUMN.
    #
    # scale_data fits a RobustScaler per column, so at T=32 a single signal gets 32
    # different medians and IQRs - which warps the very time axis a recurrent encoder
    # is trying to read. Here every lag column of a signal shares one scaler, fit on
    # all of that signal's lags stacked together.
    groups = {}
    for c in feature_names:
        base = c.rpartition("_lag")[0] if "_lag" in c else c
        groups.setdefault(base, []).append(c)

    Xtr = tr[feature_names].to_numpy(float).copy()
    Xva = va[feature_names].to_numpy(float).copy()
    Xte = te[feature_names].to_numpy(float).copy()
    pos = {c: i for i, c in enumerate(feature_names)}

    for cols in groups.values():
        idx = [pos[c] for c in cols]
        sc = RobustScaler().fit(Xtr[:, idx].reshape(-1, 1))
        for i in idx:
            for A in (Xtr, Xva, Xte):
                A[:, i] = sc.transform(A[:, i].reshape(-1, 1)).ravel()

    ytr, yva, yte = (f[target_col].to_numpy(float).reshape(-1, 1) for f in (tr, va, te))
    y_scaler = RobustScaler().fit(ytr)
    D = lambda a: pd.DataFrame(a, columns=feature_names)
    Dy = lambda a: pd.DataFrame(y_scaler.transform(a).ravel(), columns=target_col)
    return D(Xtr), D(Xva), D(Xte), Dy(ytr), Dy(yva), Dy(yte), y_scaler


def scale_param(df_train, cols, param_dict):
    stacked = pd.concat([df_train[c] for c in cols], axis=0,
                        ignore_index=True).to_numpy().reshape(-1, 1)
    sc = RobustScaler().fit(stacked)
    return {k: float(sc.transform(np.array([[v]], float))[0, 0]) for k, v in param_dict.items()}


REGELZONEN = load_data(DATA_PATH)
DF_TRAIN, DF_VAL, DF_TEST = split_data(REGELZONEN, TRAIN_RANGE, VAL_RANGE, TEST_RANGE)

print(f"rows  train {len(DF_TRAIN):>7,}   val {len(DF_VAL):>6,}   test {len(DF_TEST):>6,}")
print(f"test window  {DF_TEST[TIME_COL].min()}  ->  {DF_TEST[TIME_COL].max()}")
print(f"test price   mean {DF_TEST[LABEL[0]].mean():7.2f}   std {DF_TEST[LABEL[0]].std():7.2f}"
      f"   max {DF_TEST[LABEL[0]].max():9.2f}")

## 4. Regulatory constants C0 – C10

These are the **real thresholds from the settlement rulebook** — a 50 MW dead band, the
200/800 MW scarcity band, a EUR 1000/MWh cap — not learned parameters. They are pushed
through the same `RobustScaler` statistics as the features they get compared against.

Freezing them is the point of the architecture: capacity a black-box model would spend
rediscovering these numbers is left free to estimate the latent market state.

In [ ]:
PARAM_PRICES     = {"C1": 5, "C2": 10, "C3": 15, "C10": 1000}
PARAM_CAPACITIES = {"C4": 50, "C5": 200, "C6": 200, "C7": 200, "C8": 800, "C9": 1000}

sp = scale_param(DF_TRAIN, FEATS_PRICES, PARAM_PRICES)
sc = scale_param(DF_TRAIN, FEATS_CAPACITIES, PARAM_CAPACITIES)

C = {"C0": 0.1, **sp, **sc}
C_ORDERED = [C[f"C{i}"] for i in range(11)]

print("scaled regulatory constants")
for i in range(11):
    raw = {**PARAM_PRICES, **PARAM_CAPACITIES}.get(f"C{i}", 0.1)
    print(f"  C{i:<2} raw {raw:>6}   scaled {C[f'C{i}']:+.4f}")

## 5. Differentiable market rules

Backpropagation needs the rulebook to be smooth, so each non-differentiable primitive is
replaced by a surrogate, and **every hard `if` becomes a softmax gate** over the branches it
chooses between — all branches are computed, then blended by learned weights conditioned on
the state the regulation conditions on.

| rule | what it is |
|---|---|
| `P_RE` | volume-weighted price of activated aFRR/mFRR reserves |
| `P_EX` | liquidity-weighted blend of day-ahead and intraday prices, with a directional markup |
| `P_SC` | cubic scarcity adder once the imbalance passes the onset threshold |
| final gate | system short → take the max; long → take the min |

**This entire section is identical for all three models.** It is imported verbatim from
MRINN and is the reason the sweep is a clean single-factor ablation.

In [ ]:
def smooth_abs(x, eps=1e-9):  return K.sqrt(K.square(x) + eps)
def smooth_sign(x):           return K.tanh(x)
def smooth_max(x, y):         return y + K.softplus(x - y)
def smooth_min(x, y):         return -smooth_max(-x, -y)

def gate_first(x):  return x[:, :1]
def gate_second(x): return x[:, 1:2]

def safe_div_pair(t, eps=1e-7):
    # NOTE: sign-unaware, exactly as upstream. A negative denominator yields a
    # wrong-signed fraction. Left as-is: matching MRINN matters more than fixing it,
    # and changing it would confound the comparison.
    P, L = t
    return P / smooth_max(K.abs(L), eps)

def get_weighted_frac_safe(numer, denom, label, hidden_units):
    return layers.Lambda(safe_div_pair, output_shape=(hidden_units,))([numer, denom])


# ---- A. balancing-energy price ------------------------------------------------
def get_P_RE(E_ap, E_mp, E_an, E_mn, V, P_ap, P_mp, P_an, P_mn, P_vp, P_vn, H):
    E_sum_pos = E_ap + E_mp
    E_sum_neg = E_an + E_mn

    P_act_pos = get_weighted_frac_safe(E_ap * P_ap + E_mp * P_mp, E_sum_pos, "RE_pos", H)
    P_act_neg = get_weighted_frac_safe(E_an * P_an + E_mn * P_mn, E_sum_neg, "RE_neg", H)

    gate_in = layers.Concatenate(name="gate_RE_in")(
        [layers.Activation("tanh")(E_sum_pos), layers.Activation("tanh")(E_sum_neg), V])
    W = layers.Dense(4, activation="softmax", name="gate_RE")(gate_in)

    onesH = K.ones_like(P_act_pos)
    return layers.Add(name="P_RE_rep")([
        layers.Multiply()([P_act_pos, W[:, 0:1] * onesH]),
        layers.Multiply()([P_act_neg, W[:, 1:2] * onesH]),
        layers.Multiply()([P_vp,      W[:, 2:3] * onesH]),
        layers.Multiply()([P_vn,      W[:, 3:4] * onesH]),
    ])


# ---- B. market-reference price -------------------------------------------------
def ramp_function(P_rep, V_rep, C4, label):
    C4_t = K.ones_like(V_rep) * K.cast(C4, V_rep.dtype)
    gate_in = layers.Concatenate(name=f"gate_ramp_in_{label}")([V_rep, C4_t])
    W = layers.Dense(3, activation="softmax", name=f"gate_ramp_{label}")(gate_in)
    onesH = K.ones_like(P_rep)
    return (-onesH * (W[:, 0:1] * onesH)
            + (V_rep / (C4_t + 1e-7)) * onesH * (W[:, 1:2] * onesH)
            + onesH * (W[:, 2:3] * onesH))


def get_P_EX(P15, L15, P60, L60, PDA, V, C0, C1, C2, C3, C4, C5, C6):
    r15, r60, rDA = (ramp_function(p, V, C4, n)
                     for p, n in ((P15, "ID15"), (P60, "ID60"), (PDA, "DA")))

    C0_t = K.cast(C0, P15.dtype)
    m1 = smooth_max(K.ones_like(P15) * K.cast(C1, P15.dtype), C0_t * smooth_abs(P15))
    m2 = smooth_max(K.ones_like(P60) * K.cast(C2, P60.dtype), C0_t * smooth_abs(P60))
    m3 = smooth_max(K.ones_like(PDA) * K.cast(C3, PDA.dtype), C0_t * smooth_abs(PDA))

    P15_mkt, P60_mkt, PDA_mkt = P15 + r15 * m1, P60 + r60 * m2, PDA + rDA * m3

    one = K.ones_like(L15)
    w15 = smooth_min(one, L15 / K.cast(C5, L15.dtype))
    w60 = smooth_min(one - w15, L60 / K.cast(C6, L60.dtype))
    wDA = one - w15 - w60

    P_EX = (P15_mkt * (w15 * K.ones_like(P15_mkt))
            + P60_mkt * (w60 * K.ones_like(P60_mkt))
            + PDA_mkt * (wDA * K.ones_like(PDA_mkt)))

    P_base = layers.Add(name="P_EX_basis")([
        layers.Multiply()([P15, w15]),
        layers.Multiply()([P60, w60]),
        layers.Multiply()([PDA, wDA]),
    ])
    return P_EX, P_base


# ---- C. scarcity function ------------------------------------------------------
def get_P_SC(P_base, V, C7, C8, C9, C10):
    aV, sV = smooth_abs(V), smooth_sign(V)
    t = lambda c: K.ones_like(aV) * K.cast(c, aV.dtype)
    C7_t, C8_t, C9_t, C10_t = t(C7), t(C8), t(C9), t(C10)
    denom = (C9_t - C7_t) + 1e-7

    term0 = P_base
    term1 = P_base + sV * C10_t * K.power((aV - C7_t) / denom, 3.0)
    term2 = P_base + sV * C10_t * K.power((C8_t - C7_t) / denom, 3.0)

    W = layers.Dense(3, activation="softmax", name="Psc_gate")(
        layers.Concatenate(name="Psc_gate_in")([aV, C7_t, C8_t]))

    return layers.Add(name="Psc_mix")([
        layers.Multiply()([term0, W[:, 0:1]]),
        layers.Multiply()([term1, W[:, 1:2]]),
        layers.Multiply()([term2, W[:, 2:3]]),
    ])

print("market-rule surrogates defined (identical for all three models)")

## 6. The three models

One builder, one branch. `_encode()` is the **entire** difference between MRINN and the
recurrent variants — everything after it is shared code operating on identically-shaped
latent vectors.

Note the parameter arithmetic, which is the whole argument:

- **Dense encoder:** first layer is `T x H + H` weights **per signal**, so the model grows
  linearly with the window — 1,799 params at `T=1` up to roughly 14,000 at `T=96`.
- **Recurrent encoder:** the same cell is re-applied at every timestep, so the count is
  `constant in T` — 5,511 (LSTM) and 4,615 (GRU) at every window length.

**Only 16 of the 17 encoders carry parameters.** `get_P_EX` never consumes `L_DA`'s
representation (it derives `w_DA = 1 - w_ID15 - w_ID60`), so Keras prunes that branch, and
`load_data` pins the column to zero anyway. The input still exists and must still be fed.
Inherited from MRINN, left as-is.

In [ ]:
# (dataframe column, model input name) - ORDER IS POSITIONAL.
# The rule blocks read these tensors by position; a permutation still trains, it just
# feeds liquidity into a price slot and silently produces a wrong model.
FEATURE_SPEC = [
    ("system_imbalance", "system_imbalance"),
    ("E_aFRR_pos", "E_aFRR_pos_in"), ("E_mFRR_pos", "E_mFRR_pos_in"),
    ("P_aFRR_pos", "P_aFRR_pos_in"), ("P_mFRR_pos", "P_mFRR_pos_in"),
    ("E_aFRR_neg", "E_aFRR_neg_in"), ("E_mFRR_neg", "E_mFRR_neg_in"),
    ("P_aFRR_neg", "P_aFRR_neg_in"), ("P_mFRR_neg", "P_mFRR_neg_in"),
    ("P_aFRR_pos_MOL", "P_VoAA_pos_in"), ("P_aFRR_neg_MOL", "P_VoAA_neg_in"),
    ("P_ID15_nemo", "P_ID15_nemo_in"), ("P_ID60_nemo", "P_ID60_nemo_in"),
    ("P_DA_nemo", "P_DA_nemo_in"),
    ("L_ID15", "L_ID15_in"), ("L_ID60", "L_ID60_in"), ("L_DA", "L_DA_in"),
]
FEATURE_BASES = [b for b, _ in FEATURE_SPEC]
INPUT_NAMES   = [n for _, n in FEATURE_SPEC]
QUANTILES     = [0.1, 0.25, 0.5, 0.75, 0.9]


def make_inputs(frame, lags, kind):
    # MRINN wants flat (N, T); the recurrent models want (N, T, 1) oldest-step-first.
    arrs = []
    for base in FEATURE_BASES:
        a = frame[[f"{base}_lag{L}" for L in lags]].to_numpy("float32")   # newest -> oldest
        if kind != "mrinn":
            a = np.ascontiguousarray(a[:, ::-1])[:, :, None]              # oldest -> newest
        arrs.append(a)
    return arrs


def assert_input_alignment(model, arrays, expect_ndim):
    names = [t.name.split(":")[0] for t in model.inputs]
    if names != INPUT_NAMES:
        bad = [(i, a, b) for i, (a, b) in enumerate(zip(names, INPUT_NAMES)) if a != b]
        raise ValueError(f"input name order mismatch at {bad}")
    for n, a in zip(names, arrays):
        if a.ndim != expect_ndim:
            raise ValueError(f"input '{n}' has shape {a.shape}, expected ndim {expect_ndim}")


def multi_quantile_pinball_loss(quantiles):
    qs = tf.reshape(tf.constant(quantiles, tf.float32), (1, -1))
    def loss(y_true, y_pred):
        y_true = tf.reshape(tf.cast(y_true, tf.float32), (-1, 1))
        e = y_true - y_pred
        return tf.reduce_mean(tf.maximum(qs * e, (qs - 1.0) * e))
    return loss


def HierarchicalQuantileHeadQ50(z, quantiles, prefix="hq"):
    # Predict the median, then step outward with softplus-constrained (strictly
    # non-negative) residuals. Quantile crossing becomes arithmetically unreachable
    # rather than something training has to learn.
    sq = sorted(quantiles)
    if 0.5 not in sq:
        raise ValueError("requires quantile 0.5")
    mid = sq.index(0.5)
    nm = lambda q: f"{int(round(q * 100)):02d}"

    med = layers.Dense(1, activation="linear", name=f"{prefix}_q{nm(0.5)}")(z)
    out = {0.5: med}

    prev = med
    for q in sq[mid + 1:]:
        r = layers.Activation("softplus", name=f"{prefix}_pos_r{nm(q)}")(
            layers.Dense(1, name=f"{prefix}_r{nm(q)}")(z))
        prev = layers.Add(name=f"{prefix}_q{nm(q)}")([prev, r]); out[q] = prev

    prev = med
    for q in reversed(sq[:mid]):
        r = layers.Activation("softplus", name=f"{prefix}_pos_r{nm(q)}")(
            layers.Dense(1, name=f"{prefix}_r{nm(q)}")(z))
        prev = layers.Subtract(name=f"{prefix}_q{nm(q)}")([prev, r]); out[q] = prev

    return layers.Concatenate(axis=-1, name=f"{prefix}_output")([out[q] for q in quantiles])


# ================== THE ONLY PART THAT DIFFERS ==================
def _encode(x, kind, H, num_layer, name):
    if kind == "mrinn":
        for i in range(num_layer):
            x = layers.Dense(H, activation="swish", name=f"{name}_dense_{i}")(x)
        return x
    Cell = layers.LSTM if kind == "lstm" else layers.GRU
    for i in range(num_layer):
        x = Cell(H, return_sequences=(i < num_layer - 1), name=f"{name}_{kind}_{i}")(x)
    return x
# ================================================================


def build_model(kind, T, H=8, num_layer=None, quantiles=QUANTILES, lr=1e-3):
    kind = kind.lower()
    assert kind in {"mrinn", "lstm", "gru"}
    # MRINN's reference config stacks two Dense layers; the recurrent models use one,
    # because two stacked recurrent layers costs ~15k params and blows the budget.
    if num_layer is None:
        num_layer = 2 if kind == "mrinn" else 1

    shape = (T,) if kind == "mrinn" else (T, 1)
    feats_in = [layers.Input(shape=shape, name=n) for n in INPUT_NAMES]
    reps = [_encode(x, kind, H, num_layer, base)
            for x, base in zip(feats_in, FEATURE_BASES)]

    (V, E_ap, E_mp, P_ap, P_mp, E_an, E_mn, P_an, P_mn,
     P_vp, P_vn, P15, P60, PDA, L15, L60, LDA) = reps

    # ---- everything below is identical for all three models ----
    P_RE = get_P_RE(E_ap, E_mp, E_an, E_mn, V, P_ap, P_mp, P_an, P_mn, P_vp, P_vn, H)
    P_EX, P_base = get_P_EX(P15, L15, P60, L60, PDA, V, *C_ORDERED[:7])
    P_SC = get_P_SC(P_base, V, *C_ORDERED[7:11])

    lo = smooth_min(smooth_min(P_RE, P_EX), P_SC)
    hi = smooth_max(smooth_max(P_RE, P_EX), P_SC)
    Wf = layers.Dense(2, activation="softmax", name="final_gate")(V)
    z = layers.Add(name="imbalance_price_rep")([
        layers.Multiply()([lo, Wf[:, 0:1]]),
        layers.Multiply()([hi, Wf[:, 1:2]]),
    ])

    out = HierarchicalQuantileHeadQ50(z, quantiles, prefix="imbalance_price_hq")

    model = models.Model(feats_in, out, name=f"{kind.upper()}_T{T}")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=multi_quantile_pinball_loss(quantiles))
    return model


# quick structural check
for k in ("mrinn", "lstm", "gru"):
    m = build_model(k, T=8)
    print(f"  {k:<6} T=8  params {m.count_params():>6,}   input {m.inputs[0].shape}")
    del m
tf.keras.backend.clear_session(); gc.collect()

## 7. Evaluation metrics

Ported from `library_imbalance/evaluation.py`. Everything is computed **after** inverse
scaling, so the numbers are in EUR/MWh.

| metric | what it measures |
|---|---|
| **AQL** | mean pinball loss over the five quantiles — overall quality of the *distribution* |
| **AQCR** | quantile crossing rate — % of predictions where a lower quantile exceeds a higher one |
| **AQCE** | coverage error — does the "80% band" actually contain the truth 80% of the time |
| **AIW** | mean interval width — a band only counts if it did not get wider to buy coverage |
| **MAE / RMSE / R²** | point accuracy, computed on the median prediction |

In [ ]:
def pinball_loss_np(y, p, q):
    e = np.asarray(y).ravel() - np.asarray(p).ravel()
    return float(np.mean(np.maximum(q * e, (q - 1.0) * e)))


def AQCR_percent(P):
    P = np.asarray(P)
    if P.shape[-1] < 2:
        return 0.0
    return float((P[..., :-1] > P[..., 1:]).mean() * 100.0)


def AQCE_percent(y, P, quantiles):
    qs = np.asarray(quantiles, float); order = np.argsort(qs)
    qs, P = qs[order], np.asarray(P)[..., order]
    errs = []
    for i in range(len(qs) // 2):
        lo, hi = P[..., i], P[..., -(i + 1)]
        cov = float(((y >= lo) & (y <= hi)).mean())
        errs.append(abs(cov - float(qs[-(i + 1)] - qs[i])))
    return float(np.mean(errs) * 100.0) if errs else 0.0


def AIW_metric(P, quantiles):
    qs = np.asarray(quantiles, float); order = np.argsort(qs)
    qs, P = qs[order], np.asarray(P)[..., order]
    w = [np.mean(P[..., -(i + 1)] - P[..., i]) for i in range(len(qs) // 2)]
    return float(np.mean(w)) if w else 0.0


def evaluate_performance(y_true_scaled, yqs_scaled, quantiles, y_scaler):
    inv = lambda a: y_scaler.inverse_transform(np.asarray(a).reshape(-1, 1)).ravel()
    y = inv(np.asarray(y_true_scaled).ravel())
    yq = [inv(a) for a in yqs_scaled]
    P = np.column_stack(yq)

    mid = quantiles.index(0.5)
    r = {"RMSE": float(root_mean_squared_error(y, yq[mid])),
         "MAE":  float(mean_absolute_error(y, yq[mid])),
         "R2":   float(r2_score(y, yq[mid]))}
    r["AQL"]  = float(np.mean([pinball_loss_np(y, p, q) for q, p in zip(quantiles, yq)]))
    r["AQCR"] = AQCR_percent(P)
    r["AQCE"] = AQCE_percent(y, P, quantiles)
    r["AIW"]  = AIW_metric(P, quantiles)
    return r, y, yq[mid]

print("metrics defined")

## 8. Experiment queue

Every entry pairs a change with a baseline it can be compared against **over the same seeds**.
Blocks are ordered by expected value, so a session that runs out of time still answers the
questions that matter most.

| block | question | runs |
|---|---|---|
| **A** | Does extra width help? `H=8` vs `H=11`, MR-GRU, 3 seeds | 6 |
| **B** | MRINN control at the same seeds | 3 |
| **C** | Does shared-lag scaling help? vs A's `H=11` rows | 3 |
| **D** | Does width help MR-LSTM too? `H=8` vs `H=10`, 2 seeds | 4 |
| **E** | Does the winner hold at `T=64`? | 2 |
| **F** | Is depth better than width? 2 layers at `H=6`, 1 seed | 2 |
| **G** | Shared-lag at `H=8` — completes the 2×2 | 3 |

**Learning-rate schedule is on for every run** (cosine decay from 1e-3), so v2 is internally
consistent. Block A's `H=8` rows are the v2 baseline; comparing them against v1's `T=32` numbers
isolates what the schedule alone bought.

Every run also records **the epoch at which validation loss bottomed** — if that sits near 50,
the whole study is undertrained and the epoch budget is the next thing to raise.

In [ ]:
# ------------------------------- configuration -------------------------------
SMOKE_TEST    = False        # True -> 3 tiny runs, ~3 min, exercises everything
BUDGET_HOURS  = 7.5          # stop launching new runs past this; 12 h session limit
EPOCHS        = 50
BATCH         = 1024
BASE_LR       = 1e-3

SEEDS3, SEEDS2 = [42, 43, 44], [42, 43]

def cfgs(model, T, H, scaling, seeds, num_layer=None, tag=None):
    # MRINN's reference config stacks TWO Dense layers; the recurrent models use one.
    # Resolving it here rather than defaulting to 1 keeps MRINN a faithful baseline.
    nl = num_layer if num_layer is not None else (2 if model == "mrinn" else 1)
    return [dict(model=model, T=T, H=H, scaling=scaling, num_layer=nl, seed=s,
                 tag=tag or f"{model}_T{T}_H{H}_{scaling}"
                     + (f"_L{nl}" if (model != "mrinn" and nl > 1) else ""))
            for s in seeds]

QUEUE = (
    # A - width, MR-GRU (the top lever) + the v2 baseline
    cfgs("gru",  32,  8, "percol", SEEDS3) +
    cfgs("gru",  32, 11, "percol", SEEDS3) +
    # B - MRINN control on the same seeds
    cfgs("mrinn",32,  8, "percol", SEEDS3) +
    # C - shared-lag scaling at the wider setting
    cfgs("gru",  32, 11, "shared", SEEDS3) +
    # D - width for MR-LSTM
    cfgs("lstm", 32,  8, "percol", SEEDS2) +
    cfgs("lstm", 32, 10, "percol", SEEDS2) +
    # E - does the winner hold at the optimum window
    cfgs("gru",  64, 11, "percol", SEEDS2) +
    # F - depth instead of width
    cfgs("gru",  32,  6, "percol", [42], num_layer=2) +
    cfgs("lstm", 32,  6, "percol", [42], num_layer=2) +
    # G - completes the width x scaling grid
    cfgs("gru",  32,  8, "shared", SEEDS3)
)

if SMOKE_TEST:
    EPOCHS, BUDGET_HOURS = 2, 0.5
    QUEUE = (cfgs("gru", 4, 8, "percol", [42]) + cfgs("gru", 4, 11, "shared", [42])
             + cfgs("mrinn", 4, 8, "percol", [42]))
    RESULTS_CSV = OUT / "smoke_tuning.csv"
    PRED_NPZ    = OUT / "smoke_tuning.npz"

print(f"{len(QUEUE)} runs queued")
print(pd.DataFrame(QUEUE).groupby(["model","T","H","scaling","num_layer"]).size()
        .rename("seeds").to_string())

In [ ]:
LABELS = {"mrinn": "MRINN", "lstm": "MR-LSTM", "gru": "MR-GRU"}

def load_results():
    return pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()

def already_done(runs, c):
    if runs.empty: return False
    m = ((runs.model == c["model"]) & (runs["T"] == c["T"]) & (runs.H == c["H"]) &
         (runs.scaling == c["scaling"]) & (runs.num_layer == c["num_layer"]) &
         (runs.seed == c["seed"]))
    return bool(m.any())

def append_run(row):
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", header=not RESULTS_CSV.exists(), index=False)

PRED = dict(np.load(PRED_NPZ, allow_pickle=False)) if PRED_NPZ.exists() else {}

# data preparation is expensive at large T, and many configs share (T, scaling),
# so cache the last two prepared sets rather than rebuilding per run
_CACHE = {}

def prepare(T, scaling):
    key = (T, scaling)
    if key in _CACHE:
        return _CACHE[key]
    lags = list(range(1, T + 1))
    tr, va, te, names = shift_data(DF_TRAIN, DF_VAL, DF_TEST, LABEL[0], FEATS, lags)
    fn = scale_data_shared_lags if scaling == "shared" else scale_data
    Xtr, Xva, Xte, ytr, yva, yte, ys = fn(tr, va, te, names, LABEL)
    D = dict(lags=lags, Xtr=Xtr, Xva=Xva, Xte=Xte, ytr=ytr, yva=yva, yte=yte,
             y_scaler=ys, stamps=te[TIME_COL].to_numpy())
    while len(_CACHE) >= 2:                       # evict oldest
        _CACHE.pop(next(iter(_CACHE)))
        gc.collect()
    _CACHE[key] = D
    return D


def run_one(c):
    D = prepare(c["T"], c["scaling"])
    set_random_seed(c["seed"])
    kind = c["model"]
    tr_in = make_inputs(D["Xtr"], D["lags"], kind)
    va_in = make_inputs(D["Xva"], D["lags"], kind)
    te_in = make_inputs(D["Xte"], D["lags"], kind)

    # cosine decay over the whole run - free, and v1 never had a schedule
    steps = int(np.ceil(len(D["Xtr"]) / BATCH)) * EPOCHS
    sched = tf.keras.optimizers.schedules.CosineDecay(BASE_LR, decay_steps=steps)

    model = build_model(kind, c["T"], H=c["H"], num_layer=c["num_layer"], lr=sched)
    assert_input_alignment(model, tr_in, expect_ndim=2 if kind == "mrinn" else 3)

    ckpt = str(CKPT_DIR / f"{c['tag']}_s{c['seed']}.weights.h5")
    cb = tf.keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", mode="min",
                                            save_best_only=True, save_weights_only=True)
    t0 = time.perf_counter()
    hist = model.fit(tr_in, D["ytr"].to_numpy("float32"),
                     validation_data=(va_in, D["yva"].to_numpy("float32")),
                     epochs=EPOCHS, batch_size=BATCH, callbacks=[cb], verbose=0)
    train_sec = time.perf_counter() - t0
    best_epoch = int(np.argmin(hist.history["val_loss"])) + 1

    model.load_weights(ckpt)
    yp = model.predict(te_in, verbose=0, batch_size=BATCH)
    yqs = [yp[:, j] for j in range(len(QUANTILES))]
    res, y_orig, y50 = evaluate_performance(D["yte"], yqs, QUANTILES, D["y_scaler"])

    row = {**{k: c[k] for k in ("model", "T", "H", "scaling", "num_layer", "seed", "tag")},
           **{k: res[k] for k in ["AQL","AQCR","AQCE","AIW","MAE","RMSE","R2"]},
           "Params": int(model.count_params()), "TrainSec": round(train_sec, 1),
           "best_epoch": best_epoch, "n_test": len(y_orig)}

    key = f"{c['tag']}_s{c['seed']}"
    PRED[f"{key}_q50"] = y50.astype("float32")
    PRED[f"actual_T{c['T']}"] = y_orig.astype("float32")
    PRED[f"stamps_T{c['T']}"] = D["stamps"].astype("datetime64[ns]").astype("int64")

    del model, tr_in, va_in, te_in, yp, yqs
    tf.keras.backend.clear_session(); gc.collect()
    return row


# --------------------------------- run ---------------------------------------
t_start = time.perf_counter()
print(f"budget {BUDGET_HOURS} h  |  {len(QUEUE)} runs  |  results -> {RESULTS_CSV}\n")
skipped = 0

for i, c in enumerate(QUEUE, 1):
    runs = load_results()
    if already_done(runs, c):
        print(f"[{i:>2}/{len(QUEUE)}] {c['tag']} s{c['seed']}  already done - skipping")
        continue

    el = (time.perf_counter() - t_start) / 3600
    if el > BUDGET_HOURS:
        skipped += 1
        continue

    row = run_one(c)
    append_run(row)
    np.savez_compressed(PRED_NPZ, **PRED)

    el = (time.perf_counter() - t_start) / 3600
    warn = "  <-- best epoch is at the ceiling, likely undertrained" if row["best_epoch"] >= EPOCHS - 1 else ""
    print(f"[{i:>2}/{len(QUEUE)}] {c['tag']:<26} s{c['seed']}  "
          f"AQL {row['AQL']:7.3f}  MAE {row['MAE']:6.2f}  RMSE {row['RMSE']:6.2f}  "
          f"params {row['Params']:>6,}  ep {row['best_epoch']:>2}  "
          f"{row['TrainSec']:>6.0f}s  [{el:.2f}/{BUDGET_HOURS} h]{warn}", flush=True)

if skipped:
    print(f"\n{skipped} runs left unstarted - budget reached. Re-run this cell to continue;")
    print("completed runs are skipped automatically.")
print(f"\nelapsed {(time.perf_counter() - t_start)/3600:.2f} h")
load_results()

## 9. Did anything actually help?

Each block is a **paired comparison over the same seeds**, reported as mean ± sd. The rule from
v1 still applies: the seed spread is ±0.11–0.26 AQL, so a change that moves the mean by less
than its own standard deviation has not been demonstrated.

In [ ]:
R = load_results()
if R.empty:
    raise SystemExit("no runs recorded yet")

R["cfg"] = R.tag
g = R.groupby(["model","T","H","scaling","num_layer"])
summary = pd.DataFrame({
    "seeds":    g.seed.nunique(),
    "AQL":      g.AQL.mean().round(3),
    "AQL_sd":   g.AQL.std().round(3),
    "MAE":      g.MAE.mean().round(2),
    "RMSE":     g.RMSE.mean().round(2),
    "R2":       g.R2.mean().round(3),
    "AQCE":     g.AQCE.mean().round(2),
    "Params":   g.Params.first(),
    "best_ep":  g.best_epoch.mean().round(0),
    "sec":      g.TrainSec.mean().round(0),
}).sort_values("AQL")
print("all configurations, best first\n")
print(summary.to_string())

In [ ]:
def effect(name, a, b, question):
    # a, b are (model, T, H, scaling, num_layer) keys; compares on their common seeds
    ra = R[(R.model==a[0])&(R["T"]==a[1])&(R.H==a[2])&(R.scaling==a[3])&(R.num_layer==a[4])]
    rb = R[(R.model==b[0])&(R["T"]==b[1])&(R.H==b[2])&(R.scaling==b[3])&(R.num_layer==b[4])]
    common = sorted(set(ra.seed) & set(rb.seed))
    if not common:
        print(f"  {name:<34} not yet run"); return
    ra = ra[ra.seed.isin(common)].sort_values("seed")
    rb = rb[rb.seed.isin(common)].sort_values("seed")
    d = ra.AQL.to_numpy() - rb.AQL.to_numpy()          # negative = a is better
    md_, sd = d.mean(), d.std(ddof=1) if len(d) > 1 else float("nan")
    if len(d) > 1 and abs(md_) > abs(sd):
        verdict = "HELPS" if md_ < 0 else "HURTS"
    else:
        verdict = "no clear effect"
    print(f"  {name:<34} {md_:+.3f} +/- {sd:.3f} AQL  over {len(d)} seed(s)   {verdict}")
    print(f"      {question}")

print("paired effects  (negative = the change improved AQL)\n")
effect("width  H=11 vs H=8  (MR-GRU)",
       ("gru",32,11,"percol",1), ("gru",32,8,"percol",1),
       "is the free capacity under the 8,900 budget worth taking?")
effect("scaling  shared vs per-column  H=11",
       ("gru",32,11,"shared",1), ("gru",32,11,"percol",1),
       "does one scaler per signal beat one per lag column?")
effect("scaling  shared vs per-column  H=8",
       ("gru",32,8,"shared",1), ("gru",32,8,"percol",1),
       "same question at the original width")
effect("width  H=10 vs H=8  (MR-LSTM)",
       ("lstm",32,10,"percol",1), ("lstm",32,8,"percol",1),
       "does the width result transfer to the LSTM cell?")
effect("depth  2 layers H=6 vs 1 layer H=8",
       ("gru",32,6,"percol",2), ("gru",32,8,"percol",1),
       "is depth a better use of the budget than width?")

print("\nundertraining check")
top = R.best_epoch.describe()
late = int((R.best_epoch >= EPOCHS - 1).sum())
print(f"  best epoch: mean {R.best_epoch.mean():.0f}, max {int(R.best_epoch.max())} of {EPOCHS}")
print(f"  runs bottoming out at the last epoch: {late} / {len(R)}"
      + ("   -> raise EPOCHS, the study is undertrained" if late > len(R)*0.3
         else "   -> 50 epochs is enough"))

In [ ]:
# ---- comparison against the v1 sweep, if it is available ----
V1 = None
for p in [OUT/"sweep_results.csv", Path("/kaggle/input").rglob("sweep_results.csv")
          if Path("/kaggle/input").exists() else []]:
    try:
        cand = p if isinstance(p, Path) else next(iter(p), None)
        if cand is not None and Path(cand).exists():
            V1 = pd.read_csv(cand); break
    except Exception:
        pass

best = summary.reset_index().iloc[0]
print(f"best configuration found: {best.model} T={best['T']} H={best.H} "
      f"{best.scaling} L{best.num_layer}")
print(f"  AQL {best.AQL:.3f} +/- {best.AQL_sd if pd.notna(best.AQL_sd) else 0:.3f}"
      f"   MAE {best.MAE:.2f}   RMSE {best.RMSE:.2f}   params {int(best.Params):,}"
      f"   {'UNDER' if best.Params < 8900 else 'OVER'} the 8,900 MLP baseline")

if V1 is not None:
    v1g = V1[V1.model=="gru"].AQL.min()
    v1m = V1[V1.model=="mrinn"].AQL.min()
    print(f"\n  v1 best MR-GRU  {v1g:.3f}   ->  change {best.AQL - v1g:+.3f} AQL "
          f"({100*(best.AQL-v1g)/v1g:+.1f}%)")
    print(f"  v1 best MRINN   {v1m:.3f}   ->  gap now {100*(best.AQL-v1m)/v1m:+.1f}%")
else:
    print("\n  (attach the v1 sweep_results.csv to compare against version 1)")

## 10. Chart — what moved

In [ ]:
COL = {"mrinn":"#a5762c","lstm":"#2a78d6","gru":"#eb6834"}
S = summary.reset_index()
TCHART = int(S["T"].mode().iloc[0])   # the window most configs were run at
S = S[S["T"] == TCHART].copy()        # compare like with like
S["name"] = (S.model.map(LABELS) + "  H=" + S.H.astype(str)
             + S.num_layer.map({1:"", 2:" x2"}) + "\n" + S.scaling)
S = S.sort_values("AQL", ascending=False)

if S.empty:
    raise SystemExit("no configurations to chart yet")

fig, ax = plt.subplots(figsize=(9.5, 0.52*len(S)+1.8))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")
y = np.arange(len(S))
err = S.AQL_sd.fillna(0).to_numpy()
ax.barh(y, S.AQL, height=0.62, color=[COL[m] for m in S.model],
        edgecolor="white", linewidth=1.4, zorder=3)
ax.errorbar(S.AQL, y, xerr=err, fmt="none", ecolor="#3d3c37", elinewidth=1.2,
            capsize=3, zorder=4)
for yi, (v, e) in enumerate(zip(S.AQL, err)):
    ax.text(v + e + 0.012, yi, f"{v:.3f}", va="center", fontsize=9, color="#12120f", zorder=5)
ax.set_yticks(y); ax.set_yticklabels(S.name, fontsize=8.5)
ax.set_xlim(min(S.AQL)-0.12, max(S.AQL)+0.12)
ax.set_xlabel("AQL  (lower is better)  -  bars show mean, whiskers +/- 1 sd across seeds")
ax.set_title(f"Configurations at T={TCHART}, ranked", fontsize=11.5, fontweight="bold",
             loc="left", pad=12)
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.6); ax.set_axisbelow(True)
for sp in ("top","right","left"): ax.spines[sp].set_visible(False)
ax.tick_params(length=0)
plt.tight_layout()
plt.savefig(OUT / "chart_tuning_ranked.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show()

R.to_csv(OUT / "tuning_results.csv", index=False)
print(f"\nartifacts in {OUT}:")
for f in sorted(OUT.glob("*.png")) + sorted(OUT.glob("*.csv")) + sorted(OUT.glob("*.npz")):
    print(f"  {f.name:<28} {f.stat().st_size/1024:>8.0f} KB")

### Reading this honestly

**A change is only demonstrated if it moves the mean by more than its own standard deviation.**
The `effect()` output applies that test directly; treat "no clear effect" as exactly that, not
as a small win.

**If nothing helps, that is a result too.** It would mean `H=8` and per-column scaling were
already near-optimal for this problem, and that the remaining gap to MRINN is set by the
one-step predictability of the market state rather than by model capacity. The right response
then is not more tuning — it is a different target (a longer horizon, where context matters
more) or a more volatile evaluation window, where the scarcity rule is actually exercised.

**Watch the `best_ep` column.** If configurations are bottoming out near epoch 50, every result
here is pessimistic and the epoch budget matters more than any architectural change.